# 04. Аэрозольная деградация видимости: пылевые бури (№1), дым/смог (№14), песчаная мгла (№15)

| № | Угроза | Эффект | Последствия |
|---|---|---|---|
| 1 | Пылевые потоки и бури | Нарушение охлаждения и оптики; перегрев, потеря ориентации | Авария/посадка, невыполнение миссии |
| 14 | Промышленный смог / дым | Ослепление датчиков, коррозия плат | Невыполнение миссии, отказ электроники |
| 15 | Песчаная пыль (мгла) | Абразивное воздействие, повреждение линз | Ухудшение ТТХ, замена оптики |

## Связь с ТЗ и ключевая задача
| Сигнал | Источник | Как используем |
|---|---|---|
| видео RGB, тепловизор (ЛВИК) | вне потоков ТЗ (подтверждено заказчиком) | пропускание, спектральный наклон, неоднородности |
| LiDAR | подтверждено заказчиком | ослабление по закону Бугера-Ламберта, независимо от камеры |
| `alt`, углы камеры | БВС-1 / подвес | геометрия съёмки: наклонная дальность до каждой строки кадра |
| `temperature` | БВС-5 | температура воздуха для модели тепловизионного канала |
| `visibility_m` | НСУ-3 | внешний прогноз видимости (слияние с бортовой оценкой) |
| выход | БВС-3, НСУ-4/5 | оценка видимости, класс аэрозоля, флаги «шлейф/очаг» |

**Главная задача для проекта.** Дым - одновременно *помеха* и *цель* детектора пожара. Поэтому недостаточно оценить «видимость плохая»: нужно отличить
1. **однородную деградацию** (мгла, пыль, смог) - снижать доверие к камере;
2. **шлейф дыма** (локальный, вытянутый, исходит от точки) - кандидат на пожар;
3. **пылевой фронт** (широкая стена с прямой кромкой, без очага) - *не* пожар;
4. **горячее пятно без дыма** (нагретые камни) - ложная тепловая тревога.

> **Допущения:** синтетические сцены по физической модели переноса излучения; параметры типов аэрозоля (показатель Ангстрема α, отношение ослабления в ЛВИК κ) заданы по порядку величины, а не измерены. Реальных данных нет.

## 1. Физическая модель
По каналам c ∈ {R, G, B}: `I_c = J_c·exp(−τ_c) + A_c·(1 − exp(−τ_c))`, `τ_c = β_c·d(y) + τ_extra,c(y,x)`; дальность видимости по Кошмидеру `V = 3,912/β_550`; спектр `β_c = β_550·(550/λ_c)^α`.
Геометрия: БВС на высоте `h` смотрит под углами `φ ∈ [φ_top, φ_bot]` ниже горизонта, строка кадра → наклонная дальность `d(y) = h / sin φ(y)` (известна по телеметрии).

| Тип | α | κ = β_ЛВИК/β_550 | Идея |
|---|---|---|---|
| мелкий (дым, смог, дымка) | 1,3-1,6 | ≈ 0,03 | сильная зависимость от длины волны, ЛВИК почти прозрачен |
| крупный (пыль, песок) | ≈ 0,3 | ≈ 0,25 | почти нейтральный спектр, заметно ослабляет и ЛВИК |

Шлейф: `τ_p = τ0·exp(−s/L)·exp(−c²/2σ(s)²)`, `σ(s) = σ0 + 0,12 s` (s - расстояние вдоль ветра, c - поперёк). Пылевой фронт: сигмоида вдоль прямой кромки. Тепловизор: `T = T_g·e^{−κτ} + T_air·(1 − e^{−κτ})`, очаг +300 К.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix

from src.synth_aerosol import AeroConfig, render, depth_profile
from src.aerosol_est import (tau_dcp, anomaly_analysis, classify_rule, spectral_features, CLASSES)
from src.aerosol_eval import (sample_cfg, prepare, classify_items, vis_table, build_table, extract)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)

In [ ]:
rng = np.random.default_rng(5)
cases = [("clear", "clear"), ("haze_fine", "мелкая дымка"), ("dust_coarse", "пыль"), ("plume_fire", "шлейф + очаг"),
         ("plume_hidden", "шлейф без очага"), ("dust_front", "пылевой фронт"), ("hot_decoy", "горячие камни")]
fig, axs = plt.subplots(3, 7, figsize=(15, 6.4))
for j, (lab, name) in enumerate(cases):
    cfg = sample_cfg(rng, 100 + j, lab, "veg", tau_plume=2.0)
    if lab == "haze_fine": cfg.vis_m = 3000
    if lab == "dust_coarse": cfg.vis_m = 3000
    r = render(cfg); d = depth_profile(cfg.alt, cfg.phi_top, cfg.phi_bot)
    an, resid = anomaly_analysis(r["rgb"], r["T"], d)
    axs[0, j].imshow(r["rgb"]); axs[0, j].set_title(f"{name}\nV={cfg.vis_m/1000:.0f} км", fontsize=9)
    axs[1, j].imshow(r["T"], cmap="inferno"); axs[2, j].imshow(resid, cmap="magma", vmin=0, vmax=1.5)
    for i in range(3): axs[i, j].axis("off")
for i, t in enumerate(("RGB", "тепловизор", "аномалия τ")): axs[i, 0].text(-.05, .5, t, transform=axs[i, 0].transAxes, rotation=90, va="center", ha="right")
plt.tight_layout(); plt.show()

## 2. Оценка видимости

| Метод | Описание |
|---|---|
| **DCP, нулевое смещение** | классический тёмный канал (He et al., 2009): `t̃ = 1 − ω·min_c min_Ω(I_c/A_c)`, β̂ - регрессия `−ln t̃` по глубине через ноль |
| **DCP + наклон по глубине** | то же, но с *свободным членом*: статическое смещение (светлая местность) уходит в intercept, информацию несёт наклон по строкам; насыщенные строки (`−ln t̃ ≥ 2,2`) отбрасываются |
| **LiDAR** | наклон `ln(I R²)` против `R` = −2β_L (Бугер-Ламберт); не зависит от местности |
| **GBM (камера)** | градиентный бустинг по признакам кадра, обучен **только на растительности**, проверяется на песке (смена домена) |
| **Слияние** | взвешивание в лог-шкале по обратной дисперсии (классический DCP + LiDAR + внешний прогноз НСУ-3 с допущением σ_ln = 0,6); веса из отдельной подвыборки по типу местности и диапазону LiDAR-оценки, ошибки считаются как средний квадрат (с учётом смещения) |

Метрика: медианная абсолютная ошибка `|ln(V̂/V)|` и доля оценок в пределах ×1,5.

In [ ]:
tune = pd.concat([vis_table(150, 61, "veg"), vis_table(150, 62, "sand")], ignore_index=True)
train_v = vis_table(300, 63, "veg")
tests = {"veg": vis_table(200, 71, "veg"), "sand": vis_table(200, 72, "sand"), "mixed": vis_table(200, 73, "mixed")}
VF = ["b_slope", "b_zero", "tau_mean", "tau_top", "tau_bot", "contrast", "grad", "sat", "dc", "rb", "Tstd", "gmean"]
gbm_v = HistGradientBoostingRegressor(max_iter=250, random_state=0).fit(train_v[VF], np.log(train_v.true_V))

def clipV(x): return np.clip(x, 100, 1e5)
def add_est(df):
    df = df.copy()
    df["V_zero"] = clipV(3.912 / df.b_zero); df["V_slope"] = clipV(3.912 / df.b_slope)
    df["V_lidar"] = clipV(3.912 / df.beta_lidar); df["V_gbm"] = clipV(np.exp(gbm_v.predict(df[VF])))
    return df
tune = add_est(tune)
DC_THR = 0.5 * (tune[tune.terrain == "veg"].dc.median() + tune[tune.terrain == "sand"].dc.median())
BINS = [0, 2000, 5000, 12000, 1e9]
def key(df): return list(zip((df.dc > DC_THR).astype(int), np.digitize(df.V_lidar, BINS[1:-1])))
tune["key"] = key(tune)
W = {}
for k, g in tune.groupby("key"):
    W[k] = (max(np.mean(np.log(g.V_zero / g.true_V) ** 2), 0.02), max(np.mean(np.log(g.V_lidar / g.true_V) ** 2), 0.02))
S_EXT = 0.36
def fuse(df):
    out = []
    for (_, r), k in zip(df.iterrows(), key(df)):
        sc, sl = W.get(k, (0.5, 0.5))
        w = np.array([1 / sc, 1 / sl, 1 / S_EXT]); v = np.log([r.V_zero, r.V_lidar, r.V_ext])
        out.append(np.exp((w * v).sum() / w.sum()))
    return np.array(out)
METHODS = ["V_zero", "V_slope", "V_lidar", "V_gbm", "V_fused", "V_ext"]
rows = []; est = {}
for name, df in tests.items():
    df = add_est(df); df["V_fused"] = fuse(df); est[name] = df
    for m in METHODS:
        e = np.abs(np.log(df[m] / df.true_V))
        rows.append(dict(местность=name, метод=m, med_abs_ln=e.median(), within_x1_5=(e < np.log(1.5)).mean()))
vt = pd.DataFrame(rows).round(3)
display(vt.pivot(index="метод", columns="местность", values="med_abs_ln").reindex(METHODS))
display(vt.pivot(index="метод", columns="местность", values="within_x1_5").reindex(METHODS))

In [ ]:
vb = [500, 1500, 3000, 6000, 12000, 40000]
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for a, (name, df) in zip(axs, est.items()):
    df = df.assign(bin=pd.cut(df.true_V, vb))
    for m in ["V_zero", "V_slope", "V_lidar", "V_gbm", "V_fused", "V_ext"]:
        e = df.groupby("bin").apply(lambda g: np.abs(np.log(g[m] / g.true_V)).median())
        a.plot(range(len(e)), e.values, marker="o", label=m)
    a.set_xticks(range(5), [f"{vb[i]/1000:.1f}-{vb[i+1]/1000:.1f}" for i in range(5)], fontsize=8); a.set_title(f"местность: {name}")
    a.set_xlabel("истинная видимость, км"); a.grid(alpha=.3)
axs[0].set_ylabel("медиана |ln(V̂/V)|"); axs[0].legend(fontsize=7); plt.ylim(0, 2.0); plt.tight_layout(); plt.show()

## 3. Тип аэрозоля: мелкий или крупный
Из строкового профиля яркости по каналам оцениваются `β̂_R, β̂_G, β̂_B` → показатель Ангстрема `α̂`; из профиля температуры (воздух по БВС-5 известен) - `β̂_ЛВИК` → `κ̂ = β̂_ЛВИК/β̂_G`. Крупные частицы дают малый α̂ и большой κ̂. Классифицируем только при достаточной оптической толщине (`τ_max = β̂·d_max` выше порога): иначе оценки спектра не информативны. Различить дым и дымку (оба «мелкие») по этим признакам **нельзя**, это проверяется ниже.

In [ ]:
allv = pd.concat([d.assign(зона=n) for n, d in est.items()], ignore_index=True)
hz = allv[allv.tau_max > 1.4].copy()
KT = 0.12
hz["pred_coarse"] = hz.kappa > KT
acc_tau = hz.groupby(pd.cut(hz.true_V, [500, 1500, 3000, 6000, 12000])).apply(lambda g: (g.pred_coarse == g.coarse).mean()).round(2)
acc_terr = hz.groupby("зона").apply(lambda g: (g.pred_coarse == g.coarse).mean()).round(2)
print("доля сцен с достаточной τ:", round(len(hz) / len(allv), 2))
display(pd.DataFrame({"точность по видимости": acc_tau}).T); display(pd.DataFrame({"точность по местности": acc_terr}).T)
sm = hz[~hz.coarse]
print("дым vs дымка (по κ̂, α̂): AUROC κ̂ =", round(roc_auc_score(sm.kind == "smoke", sm.kappa), 2), "| AUROC α̂ =", round(roc_auc_score(sm.kind == "smoke", sm.alpha), 2), "(0,5 = неразличимы)")
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for k, col in (("haze", "C0"), ("smoke", "C1"), ("dust", "C3")):
    g = hz[hz.kind == k]; axs[0].scatter(g.alpha.clip(-3, 5), g.kappa.clip(0, 0.6), s=10, c=col, label=k, alpha=.6)
axs[0].axhline(KT, c="k", ls=":"); axs[0].set_xlabel("α̂ (показатель Ангстрема)"); axs[0].set_ylabel("κ̂ (ЛВИК / видимый)"); axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].hist([hz[~hz.coarse].kappa.clip(0, .6), hz[hz.coarse].kappa.clip(0, .6)], bins=30, label=["мелкий", "крупный"], stacked=False); axs[1].axvline(KT, c="k", ls=":")
axs[1].set_xlabel("κ̂"); axs[1].legend(); plt.tight_layout(); plt.show()

## 4. Шлейф дыма или однородная деградация?
Метод: из карты оптической толщины `−ln t̃` (тёмный канал) вычитается фон по строке (25-й перцентиль: шлейф и фронт его поднимают); после сглаживания выделяется **крупнейшая связная область аномалии**. Признаки: площадь, охват по осям кадра, вытянутость (отношение осей эллипса). Из теплового канала: пятна с избытком ≥ 8 К над строкой; «очаг у шлейфа» - расстояние от пятна до области ≤ 8 px.

Правило (zero-shot):
- область широкая (охват ≥ 70 % кадра по оси или площадь > 30 %) → **пылевой фронт**;
- область вытянута (≥ 2) и узкая → **шлейф**: с очагом у основания - `plume_fire`, без очага - `plume_hidden`;
- нет области, есть тепловое пятно → **горячее пятно без дыма** (`hot_decoy`);
- нет области и нет пятна: однородный аэрозоль (τ_max > 1,4) - мелкий / крупный по κ̂, иначе `clear`.

Пороги аномалии подбираются на **отдельной** выборке (зёрна 4040+), оценка на новых зёрнах.

In [ ]:
tune_items = prepare(CLASSES, 30, 4040, "veg")
tl = np.array([x[0] for x in tune_items]); rows = []
for thr in (0.2, 0.25, 0.3, 0.4, 0.5):
    p = classify_items(tune_items, thr=thr)
    neg = np.isin(tl, ["clear", "haze_fine", "dust_coarse", "hot_decoy"])
    rows.append(dict(thr=thr, acc=(p == tl).mean(), plume_fire_recall=(p[tl == "plume_fire"] == "plume_fire").mean(),
                     ложный_шлейф=np.isin(p[neg], ["plume_fire", "plume_hidden"]).mean()))
tt = pd.DataFrame(rows).round(3); display(tt)
THR = float(tt[tt.ложный_шлейф <= 0.01].sort_values("plume_fire_recall", ascending=False).thr.iloc[0]); print("выбрано thr =", THR)

In [ ]:
FC = ["beta_dcp", "kappa", "alpha", "tau_max", "beta_lidar", "an_comp_area", "an_bbox_w", "an_bbox_h", "an_elong",
      "an_comp_mean", "an_hot_n", "an_hot_max", "an_hot_near_root"]
tr = build_table(CLASSES, 80, 81, "veg")
gbm_c = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(tr[FC], tr.label)
sets = {"veg": prepare(CLASSES, 60, 91, "veg"), "sand": prepare(CLASSES, 60, 92, "sand")}
res = {}
for name, items in sets.items():
    y = np.array([x[0] for x in items]); p_rule = classify_items(items, thr=THR)
    tab = build_table(CLASSES, 60, 91 if name == "veg" else 92, name)
    p_gbm = gbm_c.predict(tab[FC]); res[name] = (y, p_rule, tab.label.values, p_gbm)
    print(f"{name}: правило acc = {(y == p_rule).mean():.3f} | GBM acc = {(tab.label.values == p_gbm).mean():.3f}")
fig, axs = plt.subplots(2, 2, figsize=(11, 9))
for i, name in enumerate(("veg", "sand")):
    y, pr, yg, pg = res[name]
    for j, (yy, pp, t) in enumerate(((y, pr, "правило"), (yg, pg, "GBM (обучен на растительности)"))):
        cm = confusion_matrix(yy, pp, labels=CLASSES, normalize="true"); a = axs[i, j]
        a.imshow(cm, vmin=0, vmax=1, cmap="Blues"); a.set_xticks(range(7), CLASSES, rotation=45, ha="right", fontsize=7); a.set_yticks(range(7), CLASSES, fontsize=7)
        for (u, v), val in np.ndenumerate(cm): a.text(v, u, f"{val:.2f}", ha="center", va="center", fontsize=7, color="k" if val < .6 else "w")
        a.set_title(f"{t}, местность: {name}", fontsize=9)
plt.tight_layout(); plt.show()

### Почему на светлой местности шлейф не виден: контраст до засветки
Дым смещает цвет пикселя к цвету засветки: `ΔI = (A − J)(1 − e^{−τ})`. Если местность по цвету близка к засветке (песок), даже толстый шлейф почти не меняет кадр. Проверим количественно и сравним три способа выделения аномалии на песке: тёмный канал, тёмный канал с эталонным кадром прошлого пролёта и прямая разность с эталоном.

In [ ]:
from src.synth_aerosol import AEROSOL, make_scene
from src.aerosol_est import tau_dcp as _t
A_sm = np.array(AEROSOL["smoke"]["A"]); rows = []
for terr in ("veg", "sand", "mixed"):
    J = np.mean([np.linalg.norm(make_scene(s, terr)[0] - A_sm, axis=2).mean() for s in range(20)])
    rows.append(dict(местность=terr, контраст_до_засветки_дыма=J))
display(pd.DataFrame(rows).set_index("местность").round(3))
items_s = sets["sand"]; ys = np.array([x[0] for x in items_s]); rows = []
for name, kw in (("тёмный канал", dict()), ("тёмный канал + эталон", dict(use_ref=True)), ("разность с эталоном", dict(use_ref=True, ref_diff=True))):
    for thr in (THR, 0.4):
        p = classify_items(items_s, thr=thr, **kw)
        rows.append({"способ": name, "thr": thr, "acc": (p == ys).mean(), "plume_fire распознан": (p[ys == "plume_fire"] == "plume_fire").mean(),
                     "ложный шлейф (негативы)": np.isin(p[np.isin(ys, ["clear", "haze_fine", "dust_coarse", "hot_decoy"])], ["plume_fire", "plume_hidden"]).mean()})
display(pd.DataFrame(rows).set_index(["способ", "thr"]).round(2))

### Абляции правила и кривая по силе шлейфа
Абляции: (а) без теплового канала (очаг не виден); (б) без геометрии фронта (признаки ширины отключены); результат считается на тех же сценах (растительность).

In [ ]:
items = sets["veg"]; y = np.array([x[0] for x in items])
ans = []
for lab, r, d, cfg in items:
    an, _ = anomaly_analysis(r["rgb"], r["T"], d, thr=THR); ans.append((an, spectral_features(r["rgb"], r["T"], d, cfg.t_air)))
def run(mod):
    return np.array([classify_rule(mod(dict(an)), sp) for an, sp in ans])
def no_thermal(a): a["hot_n"] = 0; a["hot_near_root"] = 0.0; return a
def no_geom(a): a["bbox_w"] = 0.0; a["bbox_h"] = 0.0; a["comp_area"] = min(a["comp_area"], 0.29); return a
V = {"полное правило": run(lambda a: a), "без тепловизора": run(no_thermal), "без геометрии фронта": run(no_geom)}
rows = []
for n, p in V.items():
    anyplume = np.isin(p, ["plume_fire", "plume_hidden"])
    rows.append({"вариант": n, "acc": (p == y).mean(), "recall шлейфа (любой)": anyplume[np.isin(y, ["plume_fire", "plume_hidden"])].mean(),
                 "front→шлейф": anyplume[y == "dust_front"].mean(), "hot_decoy→очаг": (p[y == "hot_decoy"] == "plume_fire").mean(),
                 "plume_fire распознан": (p[y == "plume_fire"] == "plume_fire").mean()})
display(pd.DataFrame(rows).set_index("вариант").round(2))

In [ ]:
rng = np.random.default_rng(777); rows = []
for tau0 in (0.3, 0.5, 1.0, 1.5, 2.0, 3.0):
    for bg in ("clear", "hazy"):
        for i in range(30):
            cfg = sample_cfg(rng, 777000 + len(rows), "plume_fire", "veg", tau_plume=tau0)
            cfg.vis_m, cfg.kind = (rng.uniform(15000, 40000), "haze") if bg == "clear" else (rng.uniform(5000, 12000), "smoke")
            r = render(cfg); d = depth_profile(cfg.alt, cfg.phi_top, cfg.phi_bot)
            an, _ = anomaly_analysis(r["rgb"], r["T"], d, thr=THR); sp = spectral_features(r["rgb"], r["T"], d, cfg.t_air)
            p = classify_rule(an, sp); rows.append(dict(tau0=tau0, фон=bg, шлейф=p in ("plume_fire", "plume_hidden"), fire=p == "plume_fire"))
sv = pd.DataFrame(rows); curve = sv.groupby(["tau0", "фон"]).шлейф.mean().unstack(); display(curve.round(2))
ax = curve.plot(marker="o", figsize=(6, 3.6)); ax.set_xlabel("пиковая оптическая толщина шлейфа τ0"); ax.set_ylabel("доля обнаруженных шлейфов"); ax.grid(alpha=.3)
ax.set_title("Обнаружение шлейфа: фон чистый / дымка"); plt.show()

## 5. Снижение риска и выход модуля
**Эффективная дальность контраста.** Контраст объекта на расстоянии `d` убывает как `exp(−β d)`; при пороге `C_min` дальность обнаружения `R_c = V·ln(1/C_min)/3,912` (для 5 % порога Кошмидера `R_c = V`; для практичного 20 %: `R_c ≈ 0,41 V`). Из неё следует высота, при которой наклонная дальность до цели не превышает `R_c`: `h ≤ R_c·sin φ`.

In [ ]:
rows = []
for Vkm in (1, 2, 5, 10, 20):
    V = Vkm * 1000; Rc = V * np.log(1 / 0.2) / 3.912
    rows.append(dict(видимость_км=Vkm, R_c_км=round(Rc / 1000, 2), h_max_phi20_м=round(Rc * np.sin(np.radians(20))), h_max_phi35_м=round(Rc * np.sin(np.radians(35))),
                     пропускание_на_1км=round(float(np.exp(-3.912 / V * 1000)), 2)))
display(pd.DataFrame(rows).set_index("видимость_км"))
def aerosol_alert(t, V_hat, cls, conf, uav_id="UAV-001"):
    sev = 1 if V_hat > 10000 else 2 if V_hat > 3000 else 3
    act = {"plume_fire": "raise_fire_candidate; confirm_with_thermal; keep_rgb_weight",
           "plume_hidden": "raise_smoke_candidate_no_source; request_zone_check; widen_search",
           "dust_front": "do_not_flag_fire; reduce_rgb_confidence; plan_detour_or_RTH",
           "hot_decoy": "thermal_anomaly_without_smoke; low_fire_confidence",
           "haze_fine": "reduce_rgb_confidence; prefer_thermal; lower_altitude_if_possible",
           "dust_coarse": "reduce_rgb_confidence; check_cooling_and_optics; plan_detour_or_RTH", "clear": "none"}[cls]
    return {"alert_id": f"aer-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": sev,
            "description": f"AEROSOL_{cls.upper()}", "visibility_est_m": int(V_hat), "sensor_source": "rgb+thermal+lidar",
            "confidence": conf, "timestamp": int(t * 1000), "recommended_action": act}
aerosol_alert(8.0, 2500, "dust_coarse", 0.8)

## 6. Выводы и ограничения
**Что показал эксперимент** (синтетика, три типа местности: растительность `veg`, песок `sand`, смесь `mixed`; подбор порогов и весов на отдельных зёрнах).

**Оценка видимости** (медиана |ln(V̂/V)|, veg / sand / mixed):
- классический DCP 0,33 / 0,47 / 0,29; «DCP с наклоном» 0,48 / 1,31 / 0,66 - **хуже классического**: наклон строкового профиля - слабый сигнал на фоне статического смещения местности;
- LiDAR 0,29 / 0,22 / 0,29 и не зависит от местности, но точен лишь при плотной дымке (до ≈ 6 км); при V > 12 км ошибка растёт (0,6-0,9);
- GBM по кадру лучший в «своём» домене (0,11 на растительности, 97 % оценок в пределах ×1,5), но **при смене местности деградирует** (песок: 0,38, лишь 54 % в пределах ×1,5);
- **слияние** (DCP + LiDAR + внешний прогноз НСУ-3) стабильно: 0,19 / 0,21 / 0,17, 83-87 % в пределах ×1,5; сам внешний прогноз с принятой ошибкой даёт 0,38-0,43.
Практически: LiDAR - основной при V < 6 км, камера - при умеренной дымке над тёмной местностью, внешний прогноз - страховка; GBM допустим только в области обучения.

**Тип аэрозоля.** Мелкий и крупный аэрозоль различаются по κ̂ (ослабление в ЛВИК) и α̂: точность 0,90-0,94 по диапазонам видимости, но только в 42 % сцен, где оптическая толщина достаточна; на песке 0,74 (смещение тёмного канала). **Дым и дымку различить нельзя**: AUROC κ̂ = 0,54, α̂ = 0,44 (оценка α̂ смещена цветом засветки). Отличить дым от смога без PM/CO-датчиков можно лишь по геометрии (шлейф с источником).

**Шлейф или однородная деградация.** На растительности правило даёт точность 0,87 (GBM 0,95): ложных шлейфов на чистых, дымных, пыльных сценах и горячих камнях нет, `plume_fire` распознаётся в 90 %. Абляции: без тепловизора шлейф с очагом неотличим от шлейфа без очага (точность 0,60); без геометрии фронта пылевой фронт принимается за шлейф чаще (0,22 против 0,15). Обнаружение шлейфа резко зависит от его силы: при τ0 ≥ 1 доля обнаружений 0,87-0,97 (и на чистом фоне, и в дымке), при τ0 ≤ 0,5 ≤ 0,13.

**Светлая местность - принципиальное ограничение.** Над песком контраст до засветки дыма вдвое меньше (0,25 против 0,60): дым почти не меняет цвет пикселя. Правило на песке разрушается (точность 0,21, GBM 0,63); эталонный кадр прошлого пролёта улучшает результат лишь частично (точность до 0,49, `plume_fire` 0,30-0,42). Вывод для проекта: над светлым грунтом обнаружение дыма в RGB ненадёжно, нужен тепловой и временной анализ.

**Снижение риска.** Дальность распознавания контраста R_c ≈ 0,41·V (порог 20 %); при V = 2 км высота при угле 20° не более ≈ 280 м. Алерт несёт оценку видимости, класс и рекомендацию: для пылевого фронта явно **не** поднимать пожарный флаг; для шлейфа с очагом - поднимать и подтверждать тепловизором.

**Ограничения**
- Параметры α и κ для типов аэрозоля заданы по порядку величины, цвет засветки и палитра местности - наши допущения; реальный дым, пыль, туман и их смеси перекрываются сильнее.
- Глубина считается по плоской земле и точной телеметрии (ошибка высоты ±8 %); рельеф без ЦМР нарушит оценку.
- Модель LiDAR упрощена (фиксированное распределение отражательной способности, без многократного рассеяния); внешний прогноз видимости (σ_ln = 0,6) - допущение.
- Один кадр без динамики: шлейф не меняется во времени, ветер постоянный; временное сглаживание и слежение за шлейфом не реализованы.
- Классы пересекаются на границах (V ≈ 8-15 км), поэтому точность по классам ограничена и самой постановкой.

**Дальше:** интеграционный ноутбук (слияние всех детекторов в индекс пригодности условий и влияние на детектор пожара); проверка на реальных кадрах дымки/дыма (RESIDE, FLAME) и на реальных облаках LiDAR (WADS/DENSE) по возможности.